# EV Purchases — V2 Push (Kaggle CPU only)
FE + tuned LGBM + CatBoost + XGBoost blend. No GPU. ~20-30 min on Kaggle CPU.
Run all cells, then Submit `/kaggle/working/submission_blend.csv`.

In [ ]:
# Kaggle already has these — keep versions pinned for reproducibility
# !pip install -q lightgbm==4.6.0 catboost==1.2.8 xgboost==2.1.4
import os, gc
import numpy as np, pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
print('cpu:', os.cpu_count())


In [ ]:
SEED=42; N_SPLITS=5
TRAIN_P='/kaggle/input/playground-series-s6e9/train.csv'
TEST_P='/kaggle/input/playground-series-s6e9/test.csv'
SUB_P='/kaggle/input/playground-series-s6e9/sample_submission.csv'
OUT='/kaggle/working/submission_blend.csv'
TARGET='Will_Buy_EV'
CAT_BASE=['Gender','City_Type','Current_Car_Type','Home_Charging_Possible','Subsidy_Available','Range_Anxiety_Level']
tr=pd.read_csv(TRAIN_P); te=pd.read_csv(TEST_P)
y=(tr[TARGET]=='Yes').astype(int).values
def add_features(df):
    import numpy as np
    d=df.copy()
    d['Income_log']=np.log1p(d['Annual_Income_USD'])
    d['TotalStations']=d['Charging_Stations_Near_Home']+d['Charging_Stations_Near_Work']
    d['StationDiff']=d['Charging_Stations_Near_Home']-d['Charging_Stations_Near_Work']
    d['Commute_per_station']=d['Daily_Commute_km']/(d['TotalStations']+1)
    d['SubsidyYes']=(d['Subsidy_Available']=='Yes').astype(int)
    d['HomeYes']=(d['Home_Charging_Possible']=='Yes').astype(int)
    d['AnxietyOrd']=d['Range_Anxiety_Level'].map({'Low':0,'Medium':1,'High':2}).astype(int)
    d['Env_x_Subsidy']=d['Environmental_Concern_Level']*d['SubsidyYes']
    d['Income_x_Env']=d['Income_log']*d['Environmental_Concern_Level']
    d['Env_x_Home']=d['Environmental_Concern_Level']*d['HomeYes']
    d['HomeSubsidy']=d['HomeYes']*2+d['SubsidyYes']
    d['City_Car']=d['City_Type'].astype(str)+'_'+d['Current_Car_Type'].astype(str)
    d['Env5']=(d['Environmental_Concern_Level']==5).astype(int)
    d['AnxietyHigh']=(d['Range_Anxiety_Level']=='High').astype(int)
    return d
trf=add_features(tr); tef=add_features(te)
CAT_LGB=CAT_BASE+['City_Car']
for c in CAT_LGB:
    trf[c]=trf[c].astype('category'); tef[c]=tef[c].astype('category')
FEATS=[c for c in trf.columns if c not in ['id',TARGET]]
X=trf[FEATS]; Xte=tef[FEATS]
XGB_FEATS=pd.get_dummies(pd.concat([trf[FEATS],tef[FEATS]],axis=0),columns=CAT_LGB,dtype='float32')
X_xgb=XGB_FEATS.iloc[:len(tr)].values; Xte_xgb=XGB_FEATS.iloc[len(tr):].values
print(tr.shape, te.shape, len(FEATS), X_xgb.shape)


In [ ]:
skf=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
oof_lgb=np.zeros(len(X)); oof_cat=np.zeros(len(X)); oof_xgb=np.zeros(len(X))
prd_lgb=np.zeros(len(Xte)); prd_cat=np.zeros(len(Xte)); prd_xgb=np.zeros(len(Xte))
lgb_params={'objective':'binary','metric':'auc','boosting_type':'gbdt','device_type':'cpu','verbosity':-1,'seed':SEED,'learning_rate':0.03,'num_leaves':127,'min_child_samples':150,'feature_fraction':0.7,'bagging_fraction':0.8,'bagging_freq':1,'lambda_l2':5.0}
for fold,(ti,vi) in enumerate(skf.split(X,y)):
    print(f'===== FOLD {fold+1}/{N_SPLITS} =====')
    dtr=lgb.Dataset(X.iloc[ti],label=y[ti],categorical_feature=CAT_LGB)
    dva=lgb.Dataset(X.iloc[vi],label=y[vi],categorical_feature=CAT_LGB)
    m=lgb.train(lgb_params,dtr,num_boost_round=3000,valid_sets=[dva],callbacks=[lgb.early_stopping(150,verbose=100),lgb.log_evaluation(200)])
    oof_lgb[vi]=m.predict(X.iloc[vi]); prd_lgb+=m.predict(Xte)/N_SPLITS
    print(f'[lgb] AUC={roc_auc_score(y[vi],oof_lgb[vi]):.5f} iter={m.best_iteration}')
    cTr=Pool(X.iloc[ti],y[ti],cat_features=CAT_LGB); cVa=Pool(X.iloc[vi],y[vi],cat_features=CAT_LGB)
    cm=CatBoostClassifier(iterations=2500,learning_rate=0.05,depth=7,l2_leaf_reg=5,random_strength=0.5,loss_function='Logloss',eval_metric='AUC',random_seed=SEED,task_type='CPU',early_stopping_rounds=150,verbose=200)
    cm.fit(cTr,eval_set=cVa,use_best_model=True)
    oof_cat[vi]=cm.predict_proba(X.iloc[vi])[:,1]; prd_cat+=cm.predict_proba(Xte)[:,1]/N_SPLITS
    print(f'[cat] AUC={roc_auc_score(y[vi],oof_cat[vi]):.5f} iter={cm.best_iteration_}')
    xm=XGBClassifier(n_estimators=3000,learning_rate=0.03,max_depth=7,min_child_weight=100,subsample=0.8,colsample_bytree=0.7,colsample_bylevel=0.7,reg_lambda=5.0,reg_alpha=0.5,tree_method='hist',device='cpu',random_state=SEED,eval_metric='auc',early_stopping_rounds=150)
    xm.fit(X_xgb[ti],y[ti],eval_set=[(X_xgb[vi],y[vi])],verbose=200)
    oof_xgb[vi]=xm.predict_proba(X_xgb[vi])[:,1]; prd_xgb+=xm.predict_proba(Xte_xgb)[:,1]/N_SPLITS
    print(f'[xgb] AUC={roc_auc_score(y[vi],oof_xgb[vi]):.5f} iter={xm.best_iteration}')
print(f"OOF lgb={roc_auc_score(y,oof_lgb):.5f} cat={roc_auc_score(y,oof_cat):.5f} xgb={roc_auc_score(y,oof_xgb):.5f}")


In [ ]:
best=(0,None)
for w1 in [0.4,0.5,0.6]:
    for w2 in [0.2,0.25,0.3]:
        w3=round(1-w1-w2,2)
        if w3<0.1: continue
        a=roc_auc_score(y,w1*oof_lgb+w2*oof_cat+w3*oof_xgb)
        print(f'w {w1}/{w2}/{w3} -> {a:.5f}')
        if a>best[0]: best=(a,(w1,w2,w3))
print('BEST',round(best[0],5),best[1])
w1,w2,w3=best[1]; blend=w1*prd_lgb+w2*prd_cat+w3*prd_xgb
sub=pd.read_csv(SUB_P); sub[TARGET]=blend
sub.to_csv(OUT,index=False)
print(OUT, sub.shape, float(sub[TARGET].mean()))
